# Instructions for running
See the README for more instructions

To run on a list of basins, make sure the these basins are listed in the *data/basin_list* file, including their corresponding csv file in data/all_sim_obs_lstm/00000000.txt

In [ ]:
#TODO 0: Install if not done yet , or us " uv sync " in terminal, You may also need to install some extra libraries if necessary
# !pip install -r -q requirements.txt     

In [ ]:
from tqdm import tqdm

In [ ]:
#TODO 1: Adapt to your reality 
DATA_P_P = r"Y:\repo_egu24\data_paper"      

#TODO 2: Choose your case. Note that 'test' if for quick local test, and does not need camels_context and data_paper_path
CAMELS_CASE = "fr"  # [us, fr, test]   


RUN_BASE="reruns"               #TODO 3: choose 

NB_SEEDS=5                      #TODO 4: set to 20 for present study                               

RUN_DIR = f"{RUN_BASE}_{CAMELS_CASE}"
ENSEMBLE_MODES = ["climatology", "hindcast", "realtime"][:(-1 if "us" in CAMELS_CASE else None)]
CONTEXT_RUNS = ["no_in_mlp", "lstm_in_mlp", "predict_lstm_error", "sma_in_mlp", "predict_sma_error"][:(1 if "fr" in CAMELS_CASE else None)]

### 1. Run models from scratch

In [ ]:
# Adapt
for o_mu in tqdm(CONTEXT_RUNS[:1]):
    # !python main.py --camels_context {CAMELS_CASE} --nb_seeds {NB_SEEDS} --param_grid hp=1,2,3,4,5,6,7 --run_dir {RUN_DIR} --data_paper_path {DATA_P_P} --other_model_use {o_mu}
    # !python main.py --camels_context {CAMELS_CASE} --nb_seeds {NB_SEEDS} --param_grid hp=1,2,3,4,5,6,7 --run_dir {RUN_DIR} --data_paper_path {DATA_P_P} --basin_list 01052500 01350140 --other_model_use {o_mu}
    !python main.py --camels_context {CAMELS_CASE} --nb_seeds {NB_SEEDS} --param_grid hp=1,2,3,4,5,6,7 --run_dir {RUN_DIR} --data_paper_path {DATA_P_P} --basins_file basins_2.txt --other_model_use {o_mu}
    

### 2.  Apply climatology evaluation on the prerun models

In [ ]:
# [--period yyyymmdd yyyymmdd ] can also be added in a proper context
for o_mu in tqdm(CONTEXT_RUNS):
    for ens_mode in ENSEMBLE_MODES:
        #!python multi_run.py --model_dir {RUN_DIR}/{o_mu} --run_mode {ens_mode} --camels_context CAMELS_CASE --n_sub 10 --data_paper_path {DATA_P_P}
        !python multi_run.py --model_dir {RUN_DIR}/{o_mu} --run_mode {ens_mode} --camels_context {CAMELS_CASE} --n_sub 10 --data_paper_path {DATA_P_P} --basins_file basins_2.txt

### 3. Re-evaluate a pre-run model
e.g.:
python main.py --run_mode apply --run_dir DIR --discr_model *hp1* --camels_context us/fr --data_paper_path PATH


In [ ]:
# !python main.py --run_mode apply --run_dir {RUN_DIR}/no_in_mlp --discr_model hp1 --camels_context {CAMELS_CASE} --data_paper_path {DATA_P_P} 
!python main.py --run_mode apply --run_dir {RUN_DIR}/no_in_mlp --discr_model hp1

### 4. After runs
- Get models info in runs/{context}/* --> runs/no_in_mlp
- Get climatology runs in *runs/{context}/run_zzzzz/climatology/ffff_basin.csv*
- Get climatology runs in *runs/{context}/run_zzzzz/hindcast/ffff_basin.csv*
- Get climatology runs in *runs/{context}/run_zzzzz/realtime/ffff_basin.csv*
- Get evaluation output in *runs/{context}/run_zzzzz/evaluate_on_best/ffff_basin.csv*


! Note: 
    - An evaluation mode is performed as default after each complete training. So the *option 3* is Exclusively optional
    - The logs of your runs are stored in LOGS/****.log, get the last, and clean at your convenience